# 04. Backpropagation и автодифференцирование

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/04_backprop.ipynb)

Код из раздела [modules/04_backprop.md](https://github.com/justxor/math-for-ai-2026/blob/main/modules/04_backprop.md#m04). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### Двухслойная сеть руками (то, что просят написать на собеседовании)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
N, D, H, K = 64, 2, 32, 3

# Данные: три спирали по 64 точки
r = np.linspace(0.2, 1.0, N)
X = np.concatenate([np.c_[r * np.sin(4 * r + 2.1 * k), r * np.cos(4 * r + 2.1 * k)] for k in range(K)])
X += 0.03 * rng.normal(size=X.shape); y = np.repeat(np.arange(K), N)

W1 = rng.normal(0, np.sqrt(2 / D), (D, H)); b1 = np.zeros(H)    # He-инициализация
W2 = rng.normal(0, np.sqrt(1 / H), (H, K)); b2 = np.zeros(K)

for step in range(3000):
    # forward
    a1 = X @ W1 + b1
    h = np.maximum(0, a1)
    z = h @ W2 + b2
    z -= z.max(1, keepdims=True)                      # стабильность
    p = np.exp(z); p /= p.sum(1, keepdims=True)
    loss = -np.log(p[np.arange(len(y)), y]).mean()

    # backward — каждая строка = локальная производная × входящий градиент
    dz = p.copy(); dz[np.arange(len(y)), y] -= 1; dz /= len(y)   # p − y
    dW2 = h.T @ dz;  db2 = dz.sum(0)
    dh = dz @ W2.T
    da1 = dh * (a1 > 0)                                # ReLU пропускает градиент там, где был активен
    dW1 = X.T @ da1; db1 = da1.sum(0)

    for P, G in [(W1, dW1), (b1, db1), (W2, dW2), (b2, db2)]:
        P -= 0.5 * G
    if step % 1000 == 0:
        print(step, round(loss, 3), (p.argmax(1) == y).mean())

### Двухслойная сеть руками (то, что просят написать на собеседовании)

In [ ]:
import torch
T = lambda a: torch.tensor(a, dtype=torch.float64, requires_grad=True)
W1t, b1t, W2t, b2t = T(W1), T(b1), T(W2), T(b2)
zt = torch.relu(torch.tensor(X) @ W1t + b1t) @ W2t + b2t
torch.nn.functional.cross_entropy(zt, torch.tensor(y)).backward()

# ручной backward в текущей точке
a1 = X @ W1 + b1; h = np.maximum(0, a1); z = h @ W2 + b2
p = np.exp(z - z.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
dz = p.copy(); dz[np.arange(len(y)), y] -= 1; dz /= len(y)
dW1 = X.T @ ((dz @ W2.T) * (a1 > 0))
print(np.abs(W1t.grad.numpy() - dW1).max())   # ~1e-17

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def activations():
    x = np.linspace(-5, 5, 400)
    sig = 1 / (1 + np.exp(-x))
    gelu = 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))
    silu = x * sig
    funcs = {
        "sigmoid": (sig, sig * (1 - sig)),
        "tanh": (np.tanh(x), 1 - np.tanh(x)**2),
        "ReLU": (np.maximum(0, x), (x > 0).astype(float)),
        "GELU": (gelu, np.gradient(gelu, x)),
        "SiLU / Swish": (silu, np.gradient(silu, x)),
    }
    fig, axes = plt.subplots(1, 5, figsize=(16, 3.4), sharey=True)
    for ax, (name, (f, d)) in zip(axes, funcs.items()):
        ax.plot(x, f, color=C[0], lw=2, label="f(x)")
        ax.plot(x, d, color=C[1], lw=1.6, ls="--", label="f′(x)")
        ax.set_title(name); ax.set_ylim(-1.5, 3)
    axes[0].legend(loc="upper left")
    fig.suptitle("Активации (сплошная) и их производные (пунктир): у sigmoid/tanh производная → 0 на краях — затухание градиента", y=1.04)
    save(fig, "activations")

activations()